In [1]:
!pip install -q -U transformers accelerate peft bitsandbytes trl datasets scikit-learn
!pip install -q -U --force-reinstall pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 21.5 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

from huggingface_hub import notebook_login
notebook_login()

Mounted at /content/drive


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map="auto")
model = PeftModel.from_pretrained(model, "/content/drive/MyDrive/stanza_ai_qwen_checkpoint")
model = model.eval()

print("Trained Qwen model ready for testing.")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Trained Qwen model ready for testing.


In [6]:
unseen_queries = [
    "Hey, my key card isn't working, can someone help?",
    "Is breakfast included in my stay?",
    "I need to push my checkout to 3pm if that's possible",
    "The wifi keeps disconnecting every few minutes",
    "Can you send up a bottle of wine and some chocolates?",
    "I think I was overcharged on my bill",
    "Do you guys have a gym?",
    "Room 405 needs fresh sheets please",
    "How far is the airport from here?",
    "Can I book a table for 6 tonight?",
]

In [7]:
def build_prompt(query):
    return f"""You are an intent classifier for a hotel AI system called Stanza AI. Given a guest or staff query, identify the responsible agent and the specific intent.

Query: {query}
Answer (JSON only):"""

In [8]:
import re, json as jsonlib

def generate_answer(prompt, max_new_tokens=60):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    generated = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return generated.strip()

def extract_json(text):
    match = re.search(r'\{.*?\}', text, re.DOTALL)
    if match:
        try:
            return jsonlib.loads(match.group())
        except:
            return None
    return None

for query in unseen_queries:
    prompt = build_prompt(query)
    raw_output = generate_answer(prompt)
    parsed = extract_json(raw_output)
    print(f"Query: {query}")
    if parsed:
        print(f"  → Agent: {parsed.get('agent_label')} | Intent: {parsed.get('intent_name')}")
    else:
        print(f"  → Could not parse: {raw_output}")
    print()

Query: Hey, my key card isn't working, can someone help?
  → Agent: HousekeepingAgent | Intent: maintenance_complaint

Query: Is breakfast included in my stay?
  → Agent: reservations | Intent: reservation_inclusion

Query: I need to push my checkout to 3pm if that's possible
  → Agent: frontdesk | Intent: frontdesk_checkout

Query: The wifi keeps disconnecting every few minutes
  → Agent: ConciergeAgent | Intent: complaint_submission

Query: Can you send up a bottle of wine and some chocolates?
  → Agent: FnBAgent | Intent: beverage_snack_order

Query: I think I was overcharged on my bill
  → Agent: ConciergeAgent | Intent: complaint_submission

Query: Do you guys have a gym?
  → Agent: ConciergeAgent | Intent: facility_information_query

Query: Room 405 needs fresh sheets please
  → Agent: HousekeepingAgent | Intent: linen_replacement_request

Query: How far is the airport from here?
  → Agent: ConciergeAgent | Intent: hotel_information_query

Query: Can I book a table for 6 tonight?